# 03 · analysis
Loads every finished run under `sweep_results/`, classifies it, draws the figures (each with a CSV of its values) and renders
`sweep_results/report.html`. Needs only numpy, pandas and matplotlib, so it runs here or on the laptop after a `git pull`.

In [ ]:
import os, sys, json, time
from pathlib import Path
HERE = Path.cwd()                       # this notebook lives in experiments/simple
assert (HERE / "runInflow.py").exists(), "run this notebook from experiments/simple (Jupyter's cwd is the notebook's folder)"
sys.path.insert(0, str(HERE))
ROOT = HERE / "sweep_results"           # results root (committed to git; warm_cache/ and smoke/ are ignored)
import runInflow as ri
from sweep import config as C, grids, launcher, analysis
print("cwd:", HERE, "| results:", ROOT)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
df = analysis.load_runs(ROOT)
df = df[df.group != "smoke"] if len(df) else df
df[["group", "name", "N_REF", "N_AVG", "N_FOURIER", "GRAD_INIT_FRAC", "SEED0", "verdict", "status", "n_iter_done", "loss_last", "force_first", "force_last", "wall_min"]] if len(df) else "no finished runs yet" 

In [ ]:
gradstats, gtab = analysis.load_gradstats(ROOT)
gtab[gtab.run_dir.str.contains("smoke") == False] if len(gtab) else "no gradstat results yet" 

In [ ]:
# all figures -> sweep_results/figures/<name>.png + .pdf + __<table>.csv
made = analysis.make_figures(ROOT)          # smoke results live in their own subfolder and are skipped here
from IPython.display import Image, display
for p in made:
    print(p.name); display(Image(filename=str(p), width=950))

In [ ]:
rep = analysis.write_report(ROOT); print("report:", rep)

In [ ]:
# one run in detail: the per-iterate arrays (see history_README.md next to it)
if len(df):
    h = analysis.load_history(df.iloc[0].run_dir)
    print({k: (v.shape if hasattr(v, "shape") else v) for k, v in h.items()})

In [ ]:
# ship the results back through git (plain files only; run when a batch is done)
# !git add sweep_results && git commit -m "sweep results $(date +%F)" && git push